# StatsPAI 复现程度：网间互联质量与合作创新（正文表 1—表 5）

姊妹篇 [sjjj2026_replication.ipynb](sjjj2026_replication.ipynb) 用作者 Stata 代码（`jwdid ..., method(ppmlhdfe)` + `estat simple, predict(xb)`）复现了正文表 1—表 5，21/21 与论文一致。
本 notebook 只用 **StatsPAI** 重做同样的 21 个回归，目标是回答两个问题：

1. StatsPAI 能否复现论文数字？用哪种写法？
2. 哪些地方 StatsPAI **做不到或需要手工补**？——这些以 <span style="color:#c0392b">**⚠ StatsPAI 不足**</span> 标出，并在文末汇总成升级清单。

**Stata 模型（jwdid 默认 not-yet-treated，PPML）**

$$E[y_{it}\mid\cdot]=\exp\Big(\alpha_i+\lambda_t+\sum_{g}\sum_{t\ge g}\beta_{gt}\,1\{G_i=g,\,T=t\}\Big),\qquad
\text{论文报告 }\widehat{ATT}_{simple}=\frac{\sum_{g,t\ge g} N_{gt}\,\hat\beta_{gt}}{\sum N_{gt}}$$

即 **线性指数（xb，对数点）尺度** 的队列×年份系数按处理观测数加权平均，城市对聚类标准误。

**StatsPAI 两条实现路线**

| 路线 | 调用 | 作用 |
|---|---|---|
| A（主） | `sp.etwfe(..., family="poisson")` | StatsPAI 原生 Wooldridge 非线性 ETWFE |
| B（交叉验证） | `sp.fepois("y ~ tr_g_t... \| pair + year")` + `.lincom()` | 手工构造 jwdid 交互项，走 pyfixest PPML-HDFE |

## 0. 环境

In [1]:
import time, warnings, platform
import numpy as np, pandas as pd
import statspai as sp
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160, "display.max_columns", 20, "display.float_format", "{:.4f}".format)
print("statspai", sp.__version__, "| python", platform.python_version(), "|", sp.__file__)

DATA = "pkg/数据复现包/data/main.dta"
df0 = pd.read_stata(DATA, convert_categoricals=False)
print(df0.shape)
print("首次处理年份（0 = 从未处理）：", df0["first_treat_lag"].value_counts().sort_index().to_dict())
print("年份：", df0["year"].min(), "—", df0["year"].max(), "| 城市对：", df0["id_ci_cj"].nunique())

statspai 1.32.0 | python 3.13.2 | /Users/brycewang/Documents/GitHub/StatsPAI/src/statspai/__init__.py


(923076, 67)
首次处理年份（0 = 从未处理）： {0: 168483, 2015: 720678, 2018: 33915}
年份： 2000 — 2020 | 城市对： 43956


## 1. 工具函数

- `run_etwfe`：路线 A。`sp.etwfe` 的 headline 是**响应尺度平均边际效应（件数）**，不是论文的 xb 尺度 ATT，
  因此需从 `model_info` 取出 $\hat\beta_{gt}$ 与协方差手工加权（delta 法）。
- `run_fepois`：路线 B。手工生成 `tr_g_t` 交互项（与 jwdid 的 `c.__tr__#i.gvar#i.year` 相同），用 `.lincom()` 聚合。

In [2]:
def tr_cells(df, gvar="first_treat_lag", tvar="year"):
    """jwdid 默认 (not-yet) 设计：每个队列 g、每个 t >= g 一个交互项。"""
    ft = df[gvar].fillna(0)
    tmax = df[tvar].max()
    names, weights = [], []
    for g in sorted(ft[ft > 0].unique()):
        if g >= tmax + 1:          # jwdid: gvar >= gvarmax 的队列视作未处理
            continue
        for t in range(int(g), int(tmax) + 1):
            c = f"tr_{int(g)}_{t}"
            df[c] = ((ft == g) & (df[tvar] == t)).astype(float)
            names.append(c)
    return names


def run_etwfe(df, y):
    """路线 A：sp.etwfe(family='poisson')，并手工聚合为 xb 尺度 simple ATT。"""
    d = df.copy(); d["_y"] = d[y].astype(float)
    t0 = time.time()
    r = sp.etwfe(d, y="_y", group="id_ci_cj", time="year",
                 first_treat="first_treat_lag", family="poisson")
    sec = time.time() - t0
    mi = r.model_info
    nm = mi["coef_names"]
    b = pd.Series(mi["coefficients"], index=nm)
    V = pd.DataFrame(mi["vcov"], index=nm, columns=nm)
    cells = [f"treat[{g},{t}]" for g, t in mi["interaction_cells"]]
    used = d.dropna(subset=["_y"])
    w = np.array([((used["first_treat_lag"] == g) & (used["year"] == t)).sum()
                  for g, t in mi["interaction_cells"]], float)
    w /= w.sum()
    # ⚠ StatsPAI 不足 #1/#2：以下两行本应由 sp.etwfe 的 link 尺度聚合完成
    att = float(w @ b[cells].values)
    se = float(np.sqrt(w @ V.loc[cells, cells].values @ w))
    return dict(b_xb=att, se_xb=se, ame=r.estimate, se_ame=r.se, N=r.n_obs, sec=sec,
                beta=b[cells])


def run_fepois(df, y):
    """路线 B：sp.fepois + 手工交互项 + .lincom() 聚合。"""
    d = df.dropna(subset=[y]).copy(); d["_y"] = d[y].astype(float)
    cells = tr_cells(d)
    t0 = time.time()
    r = sp.fepois("_y ~ " + " + ".join(cells) + " | id_ci_cj + year", data=d,
                  vcov={"CRV1": "id_ci_cj"})
    sec = time.time() - t0
    n = np.array([d[c].sum() for c in cells]); w = n / n.sum()
    lc = r.lincom(" + ".join(f"{wi:.15f}*{c}" for wi, c in zip(w, cells)))
    return dict(b_fe=lc["estimate"], se_fe=lc["se"], N_fe=int(r.data_info["nobs"]), sec_fe=sec)


def compare(title, outcomes, paper, subsets=None):
    """对一张表的每列跑 A、B 两条路线，与论文数值对比（4 位小数）。"""
    rows = []
    for k, (y, (pb, ps, pn)) in enumerate(zip(outcomes, paper), 1):
        d = subsets[k] if subsets and k in subsets else df0
        a = run_etwfe(d, y); bb = run_fepois(d, y)
        ok = lambda v, p: abs(round(v, 4) - p) < 1e-9
        rows.append({
            "列": f"({k}) {y}", "论文 b": pb, "论文 se": ps, "论文 N": pn,
            "A etwfe-xb b": a["b_xb"], "A se": a["se_xb"], "A N": a["N"],
            "B fepois b": bb["b_fe"], "B se": bb["se_fe"], "B N": bb["N_fe"],
            "etwfe 原生AME": a["ame"],
            "A 一致": "YES" if ok(a["b_xb"], pb) and ok(a["se_xb"], ps) and a["N"] == pn else "NO",
            "B b 一致": "YES" if ok(bb["b_fe"], pb) else "NO",
            "B b+se 一致": "YES" if ok(bb["b_fe"], pb) and ok(bb["se_fe"], ps) else "NO",
            "秒 A/B": f"{a['sec']:.0f}/{bb['sec_fe']:.0f}",
        })
    out = pd.DataFrame(rows).set_index("列")
    print(title)
    print(f"路线 A 完全一致 {(out['A 一致']=='YES').sum()}/{len(out)}；"
          f"路线 B：系数一致 {(out['B b 一致']=='YES').sum()}/{len(out)}，系数+SE 一致 {(out['B b+se 一致']=='YES').sum()}/{len(out)}（N 口径不同，见第 8 节）")
    return out

RESULTS = {}

## 2. 先看一列：表 1 列 (1)，逐系数对照 Stata

Stata `jwdid` 输出的 9 个队列×年份系数（见姊妹 notebook）：

In [3]:
stata_beta = pd.Series({  # 来自 Stata jwdid 输出（合作专利）
    "treat[2015,2015]": -.1165515, "treat[2015,2016]": -.0763155, "treat[2015,2017]": .1397608,
    "treat[2015,2018]":  .2253192, "treat[2015,2019]":  .3784335, "treat[2015,2020]": .4945476,
    "treat[2018,2018]": -.1182973, "treat[2018,2019]": -.0233922, "treat[2018,2020]": -.000384})
stata_se = pd.Series([.0742705, .0738444, .0809341, .0826994, .0767525, .0750585, .2760284, .2300744, .2915038],
                     index=stata_beta.index)

d = df0.copy(); d["_y"] = d["合作专利"].astype(float)
r = sp.etwfe(d, y="_y", group="id_ci_cj", time="year", first_treat="first_treat_lag", family="poisson")
mi = r.model_info
b = pd.Series(mi["coefficients"], index=mi["coef_names"])[stata_beta.index]
se = pd.Series(np.sqrt(np.diag(mi["vcov"])), index=mi["coef_names"])[stata_beta.index]
pd.DataFrame({"Stata b": stata_beta, "StatsPAI b": b, "Δb": b - stata_beta,
              "Stata se": stata_se, "StatsPAI se": se, "Δse/se": se / stata_se - 1}).style.format("{:.2e}", subset=["Δb", "Δse/se"]).format("{:.6f}", subset=["Stata b", "StatsPAI b", "Stata se", "StatsPAI se"])

,Stata b,StatsPAI b,Δb,Stata se,StatsPAI se,Δse/se
"treat[2015,2015]",-0.116552,-0.116551,1.91e-08,0.074271,0.074272,1.65e-05
"treat[2015,2016]",-0.076315,-0.076316,-4.73e-08,0.073844,0.073846,1.72e-05
"treat[2015,2017]",0.139761,0.139761,-2.99e-08,0.080934,0.080935,1.73e-05
"treat[2015,2018]",0.225319,0.225319,3.99e-08,0.082699,0.082701,1.63e-05
"treat[2015,2019]",0.378433,0.378433,-9.96e-09,0.076753,0.076754,1.62e-05
"treat[2015,2020]",0.494548,0.494548,-1.75e-08,0.075059,0.075060,1.67e-05
"treat[2018,2018]",-0.118297,-0.118297,2.99e-08,0.276028,0.276033,1.69e-05
"treat[2018,2019]",-0.023392,-0.023392,3.64e-08,0.230074,0.230078,1.68e-05
"treat[2018,2020]",-0.000384,-0.000384,-5.09e-09,0.291504,0.291509,1.67e-05


In [4]:
print(r.summary() if hasattr(r, "summary") else r)
print("\nsp.etwfe headline estimate =", round(r.estimate, 4), "（se", round(r.se, 4), "）")
print("论文 / Stata estat simple, predict(xb) =", 0.1691, "（se", 0.0634, "）")

  Wooldridge (2023) nonlinear ETWFE — family=poisson

  ATT (average marginal effect, response scale):       0.142 ***
  Std. Error:  (0.042)
  [95% CI]:    [0.060,  0.223]
  P-value:     <0.001

------------------------------------------------------------------------------
  Event Study Coefficients
------------------------------------------------------------------------------
  e =   0  |     -0.0667  (0.0488) 
  e =   1  |     -0.0386  (0.0470) 
  e =   2  |      0.0744  (0.0629) 
  e =   3  |      0.1459  (0.0505) ***
  e =   4  |      0.2904  (0.0521) ***
  e =   5  |      0.4673  (0.0614) ***

------------------------------------------------------------------------------
  Group-Time ATT Estimates
------------------------------------------------------------------------------
    att     se
 0.1495 0.0390
-0.1811 1.1612

------------------------------------------------------------------------------
  Observations:    923,076
  Estimator:    etwfe_glm
  Family:    poisson
  Link:  

<span style="color:#c0392b">**⚠ StatsPAI 不足 #1（估计量口径，最重要）**</span>：
`sp.etwfe(family="poisson")` 的 9 个 $\hat\beta_{gt}$ 与 Stata 一致到 1e-6 以上，但 **headline 只提供响应尺度平均边际效应（AME，件数）= 0.1419**，
没有 Stata `estat simple, predict(xb)`（也即 R `etwfe::emfx(..., type="simple", predict="link")`）的 **link/xb 尺度** 聚合 0.1691。
用户若直接用 `r.estimate` 会得到与论文不同的数字（且单位不同）。

**建议升级**：`sp.etwfe(..., family="poisson", scale="link"|"response")` 或 `r.aggregate(type="simple"|"group"|"calendar"|"event", scale="link")`。

<span style="color:#c0392b">**⚠ StatsPAI 不足 #2**</span>：非线性分支没有公开的聚合 API（Stata 的 `estat simple/group/calendar/event`），
只能从 `model_info["coefficients"]`、`model_info["vcov"]`、`model_info["interaction_cells"]` 手工做 delta 法；且 `model_info` 未提供每个单元格的处理观测数 $N_{gt}$，需要回到原始数据自己数。

## 3. 表 1　新增骨干直联点开通与城市间合作专利

| 列 | 论文 b | 论文 se | 论文 N |
|---|---|---|---|
| (1) `合作专利` | 0.1691 | (0.0634) | 923,076 |
| (2) `合作外观设计专利` | -0.1316 | (0.2019) | 923,076 |
| (3) `合作实用新型专利` | 0.1812 | (0.0758) | 923,076 |
| (4) `合作发明专利` | 0.1978 | (0.0750) | 923,076 |

In [5]:
RESULTS["T1"] = compare("表 1　新增骨干直联点开通与城市间合作专利", ["合作专利", "合作外观设计专利", "合作实用新型专利", "合作发明专利"],
    [(0.1691, 0.0634, 923076), (-0.1316, 0.2019, 923076), (0.1812, 0.0758, 923076), (0.1978, 0.075, 923076)])
RESULTS["T1"]

表 1　新增骨干直联点开通与城市间合作专利
路线 A 完全一致 4/4；路线 B：系数一致 4/4，系数+SE 一致 3/4（N 口径不同，见第 8 节）


,论文 b,论文 se,论文 N,A etwfe-xb b,A se,A N,B fepois b,B se,B N,etwfe 原生AME,A 一致,B b 一致,B b+se 一致,秒 A/B
列,,,,,,,,,,,,,,
(1) 合作专利,0.1691,0.0634,923076,0.1691,0.0634,923076,0.1691,0.0634,209433,0.1419,YES,YES,YES,6/8
(2) 合作外观设计专利,-0.1316,0.2019,923076,-0.1316,0.2019,923076,-0.1316,0.2020,33348,-0.0023,YES,YES,NO,7/1
(3) 合作实用新型专利,0.1812,0.0758,923076,0.1812,0.0758,923076,0.1812,0.0758,158592,0.0579,YES,YES,YES,6/2
(4) 合作发明专利,0.1978,0.0750,923076,0.1978,0.0750,923076,0.1978,0.0750,177870,0.0944,YES,YES,YES,6/2


## 4. 表 2　新增骨干直联点开通与跨城产学研合作

| 列 | 论文 b | 论文 se | 论文 N |
|---|---|---|---|
| (1) `产学研1` | 0.1955 | (0.0924) | 923,076 |
| (2) `产学研2` | 0.2170 | (0.1207) | 923,076 |
| (3) `产学研3` | 0.1609 | (0.0966) | 923,076 |
| (4) `产学研4` | 0.1598 | (0.0967) | 923,076 |

In [6]:
RESULTS["T2"] = compare("表 2　新增骨干直联点开通与跨城产学研合作", ["产学研1", "产学研2", "产学研3", "产学研4"],
    [(0.1955, 0.0924, 923076), (0.217, 0.1207, 923076), (0.1609, 0.0966, 923076), (0.1598, 0.0967, 923076)])
RESULTS["T2"]

表 2　新增骨干直联点开通与跨城产学研合作
路线 A 完全一致 4/4；路线 B：系数一致 4/4，系数+SE 一致 4/4（N 口径不同，见第 8 节）


,论文 b,论文 se,论文 N,A etwfe-xb b,A se,A N,B fepois b,B se,B N,etwfe 原生AME,A 一致,B b 一致,B b+se 一致,秒 A/B
列,,,,,,,,,,,,,,
(1) 产学研1,0.1955,0.0924,923076,0.1955,0.0924,923076,0.1955,0.0924,106911,0.0269,YES,YES,YES,6/1
(2) 产学研2,0.2170,0.1207,923076,0.2170,0.1207,923076,0.2170,0.1207,122577,0.0406,YES,YES,YES,6/1
(3) 产学研3,0.1609,0.0966,923076,0.1609,0.0966,923076,0.1609,0.0966,99792,0.0186,YES,YES,YES,6/1
(4) 产学研4,0.1598,0.0967,923076,0.1598,0.0967,923076,0.1598,0.0967,99561,0.0185,YES,YES,YES,6/1


## 5. 表 3　既有合作、合作进入与新知识组合

| 列 | 论文 b | 论文 se | 论文 N |
|---|---|---|---|
| (1) `合作发明专利` | 0.0517 | (0.0752) | 77,952 |
| (2) `合作发明专利激活` | 0.4016 | (0.0273) | 923,076 |
| (3) `首次合作` | 0.5127 | (0.0486) | 923,076 |
| (4) `新增合作申请人数量` | 0.2671 | (0.0237) | 923,076 |
| (5) `新IPC组合` | 0.2988 | (0.0541) | 923,076 |

列 (1) 样本限定为政策前（≤2013）已发生合作发明的城市对（Stata：`bysort id: egen pre_coop = max(合作发明专利>0 & year<=2013)`）。

In [7]:
pre = (df0.assign(_p=(df0["合作发明专利"] > 0) & (df0["year"] <= 2013))
         .groupby("id_ci_cj")["_p"].transform("max"))
subsets = {1: df0[pre == 1].copy()}
print("政策前已有合作样本：", len(subsets[1]))
RESULTS["T3"] = compare("表 3　既有合作、合作进入与新知识组合", ["合作发明专利", "合作发明专利激活", "首次合作", "新增合作申请人数量", "新IPC组合"],
    [(0.0517, 0.0752, 77952), (0.4016, 0.0273, 923076), (0.5127, 0.0486, 923076), (0.2671, 0.0237, 923076), (0.2988, 0.0541, 923076)], subsets=subsets)
RESULTS["T3"]

政策前已有合作样本： 77952


表 3　既有合作、合作进入与新知识组合
路线 A 完全一致 5/5；路线 B：系数一致 5/5，系数+SE 一致 5/5（N 口径不同，见第 8 节）


,论文 b,论文 se,论文 N,A etwfe-xb b,A se,A N,B fepois b,B se,B N,etwfe 原生AME,A 一致,B b 一致,B b+se 一致,秒 A/B
列,,,,,,,,,,,,,,
(1) 合作发明专利,0.0517,0.0752,77952,0.0517,0.0752,77952,0.0517,0.0752,77952,0.5053,YES,YES,YES,0/0
(2) 合作发明专利激活,0.4016,0.0273,923076,0.4016,0.0273,923076,0.4016,0.0273,177870,0.0208,YES,YES,YES,5/2
(3) 首次合作,0.5127,0.0486,923076,0.5127,0.0486,923076,0.5127,0.0486,177870,0.0061,YES,YES,YES,6/2
(4) 新增合作申请人数量,0.2671,0.0237,923076,0.2671,0.0237,923076,0.2671,0.0237,177870,0.0454,YES,YES,YES,6/2
(5) 新IPC组合,0.2988,0.0541,923076,0.2988,0.0541,923076,0.2988,0.0541,128835,0.0641,YES,YES,YES,6/1


## 6. 表 4　嵌入对方独有优势技术：合作内容层证据

| 列 | 论文 b | 论文 se | 论文 N |
|---|---|---|---|
| (1) `RemoteEmbedCount1` | 0.1785 | (0.0767) | 922,477 |
| (2) `RemoteEmbedCount2` | 0.7114 | (0.3106) | 922,477 |
| (3) `RemoteIntensity` | 0.3863 | (0.0336) | 922,477 |

In [8]:
RESULTS["T4"] = compare("表 4　嵌入对方独有优势技术：合作内容层证据", ["RemoteEmbedCount1", "RemoteEmbedCount2", "RemoteIntensity"],
    [(0.1785, 0.0767, 922477), (0.7114, 0.3106, 922477), (0.3863, 0.0336, 922477)])
RESULTS["T4"]

表 4　嵌入对方独有优势技术：合作内容层证据
路线 A 完全一致 0/3；路线 B：系数一致 3/3，系数+SE 一致 2/3（N 口径不同，见第 8 节）


,论文 b,论文 se,论文 N,A etwfe-xb b,A se,A N,B fepois b,B se,B N,etwfe 原生AME,A 一致,B b 一致,B b+se 一致,秒 A/B
列,,,,,,,,,,,,,,
(1) RemoteEmbedCount1,0.1785,0.0767,922477,0.1829,0.0768,922477,0.1785,0.0767,140673,0.0498,NO,YES,YES,6/1
(2) RemoteEmbedCount2,0.7114,0.3106,922477,0.7163,0.3110,922477,0.7114,0.3108,38375,0.0071,NO,YES,NO,7/1
(3) RemoteIntensity,0.3863,0.0336,922477,0.3844,0.0336,922477,0.3863,0.0336,140673,0.0091,NO,YES,YES,5/1


## 7. 表 5　新增骨干直联点开通与知识融合及数实融合

| 列 | 论文 b | 论文 se | 论文 N |
|---|---|---|---|
| (1) `合作人数` | 0.4361 | (0.0298) | 923,076 |
| (2) `合作发明专利知识宽度` | 0.4776 | (0.0353) | 922,477 |
| (3) `ratioincross` | 0.4033 | (0.0369) | 923,076 |
| (4) `ratioitincross` | 0.5165 | (0.0746) | 923,076 |
| (5) `r融合专利` | 0.4035 | (0.0402) | 923,076 |

In [9]:
RESULTS["T5"] = compare("表 5　新增骨干直联点开通与知识融合及数实融合", ["合作人数", "合作发明专利知识宽度", "ratioincross", "ratioitincross", "r融合专利"],
    [(0.4361, 0.0298, 923076), (0.4776, 0.0353, 922477), (0.4033, 0.0369, 923076), (0.5165, 0.0746, 923076), (0.4035, 0.0402, 923076)])
RESULTS["T5"]

表 5　新增骨干直联点开通与知识融合及数实融合
路线 A 完全一致 4/5；路线 B：系数一致 5/5，系数+SE 一致 5/5（N 口径不同，见第 8 节）


,论文 b,论文 se,论文 N,A etwfe-xb b,A se,A N,B fepois b,B se,B N,etwfe 原生AME,A 一致,B b 一致,B b+se 一致,秒 A/B
列,,,,,,,,,,,,,,
(1) 合作人数,0.4361,0.0298,923076,0.4361,0.0298,923076,0.4361,0.0298,177870,0.0642,YES,YES,YES,5/2
(2) 合作发明专利知识宽度,0.4776,0.0353,922477,0.4763,0.0352,922477,0.4776,0.0353,149755,0.0083,NO,YES,YES,5/1
(3) ratioincross,0.4033,0.0369,923076,0.4033,0.0369,923076,0.4033,0.0369,122388,0.0091,YES,YES,YES,5/1
(4) ratioitincross,0.5165,0.0746,923076,0.5165,0.0746,923076,0.5165,0.0746,59178,0.0030,YES,YES,YES,5/1
(5) r融合专利,0.4035,0.0402,923076,0.4035,0.0402,923076,0.4035,0.0402,101094,0.0068,YES,YES,YES,5/1


## 8. 汇总：21 个回归的复现程度

In [10]:
allres = pd.concat(RESULTS, names=["表"])
summary = allres.groupby(level=0).agg(列数=("A 一致", "size"),
                                      A_一致=("A 一致", lambda s: (s == "YES").sum()),
                                      B_b一致=("B b 一致", lambda s: (s == "YES").sum()),
                                      B_bse一致=("B b+se 一致", lambda s: (s == "YES").sum()))
summary.loc["合计"] = summary.sum()
print(summary)
print("\n最大 |Δb|（A 路线 vs 论文）：", (allres["A etwfe-xb b"] - allres["论文 b"]).abs().max().round(6))
print("最大 |Δse|（A 路线 vs 论文）：", (allres["A se"] - allres["论文 se"]).abs().max().round(6))
print("最大 |Δb|（B 路线 vs 论文）：", (allres["B fepois b"] - allres["论文 b"]).abs().max().round(6))
print("最大 |Δse|（B 路线 vs 论文）：", (allres["B se"] - allres["论文 se"]).abs().max().round(6))
print("\n若误用 sp.etwfe 原生 headline（AME）：与论文差异最大",
      (allres["etwfe 原生AME"] - allres["论文 b"]).abs().max().round(4))
allres[["论文 b", "A etwfe-xb b", "B fepois b", "etwfe 原生AME", "论文 se", "A se", "B se", "论文 N", "B N", "A 一致", "B b 一致", "B b+se 一致"]]

    列数  A_一致  B_b一致  B_bse一致
表                           
T1   4     4      4        3
T2   4     4      4        4
T3   5     5      5        5
T4   3     0      3        2
T5   5     4      5        5
合计  21    17     21       19

最大 |Δb|（A 路线 vs 论文）： 0.004898
最大 |Δse|（A 路线 vs 论文）： 0.000402
最大 |Δb|（B 路线 vs 论文）： 4.8e-05
最大 |Δse|（B 路线 vs 论文）： 0.000232

若误用 sp.etwfe 原生 headline（AME）：与论文差异最大 0.7043


论文 b  A etwfe-xb b  B fepois b  etwfe 原生AME  论文 se   A se   B se    论文 N     B N A 一致 B b 一致 B b+se 一致
表  列                                                                                                                              
T1 (1) 合作专利               0.1691        0.1691      0.1691       0.1419 0.0634 0.0634 0.0634  923076  209433  YES    YES       YES
   (2) 合作外观设计专利          -0.1316       -0.1316     -0.1316      -0.0023 0.2019 0.2019 0.2020  923076   33348  YES    YES        NO
   (3) 合作实用新型专利           0.1812        0.1812      0.1812       0.0579 0.0758 0.0758 0.0758  923076  158592  YES    YES       YES
   (4) 合作发明专利             0.1978        0.1978      0.1978       0.0944 0.0750 0.0750 0.0750  923076  177870  YES    YES       YES
T2 (1) 产学研1               0.1955        0.1955      0.1955       0.0269 0.0924 0.0924 0.0924  923076  106911  YES    YES       YES
   (2) 产学研2               0.2170        0.2170      0.2170       0.0406 0.1207 0.1207 0.1207  923076  122577  YES    YES       YES
   (3) 产学研3               0.1609        0.1609      0.1609       0.0186 0.0966 0.0966 0.0966  923076   99792  YES    YES       YES
   (4) 产学研4               0.1598        0.1598      0.1598       0.0185 0.0967 0.0967 0.0967  923076   99561  YES    YES       YES
T3 (1) 合作发明专利             0.0517        0.0517      0.0517       0.5053 0.0752 0.0752 0.0752   77952   77952  YES    YES       YES
   (2) 合作发明专利激活           0.4016        0.4016      0.4016       0.0208 0.0273 0.0273 0.0273  923076  177870  YES    YES       YES
   (3) 首次合作               0.5127        0.5127      0.5127       0.0061 0.0486 0.0486 0.0486  923076  177870  YES    YES       YES
   (4) 新增合作申请人数量          0.2671        0.2671      0.2671       0.0454 0.0237 0.0237 0.0237  923076  177870  YES    YES       YES
   (5) 新IPC组合             0.2988        0.2988      0.2988       0.0641 0.0541 0.0541 0.0541  923076  128835  YES    YES       YES
T4 (1) RemoteEmbedCount1  0.1785        0.1829      0.1785       0.0498 0.0767 0.0768 0.0767  922477  140673   NO    YES       YES
   (2) RemoteEmbedCount2  0.7114        0.7163      0.7114       0.0071 0.3106 0.3110 0.3108  922477   38375   NO    YES        NO
   (3) RemoteIntensity    0.3863        0.3844      0.3863       0.0091 0.0336 0.0336 0.0336  922477  140673   NO    YES       YES
T5 (1) 合作人数               0.4361        0.4361      0.4361       0.0642 0.0298 0.0298 0.0298  923076  177870  YES    YES       YES
   (2) 合作发明专利知识宽度         0.4776        0.4763      0.4776       0.0083 0.0353 0.0352 0.0353  922477  149755   NO    YES       YES
   (3) ratioincross       0.4033        0.4033      0.4033       0.0091 0.0369 0.0369 0.0369  923076  122388  YES    YES       YES
   (4) ratioitincross     0.5165        0.5165      0.5165       0.0030 0.0746 0.0746 0.0746  923076   59178  YES    YES       YES
   (5) r融合专利              0.4035        0.4035      0.4035       0.0068 0.0402 0.0402 0.0402  923076  101094  YES    YES       YES

### 8.1 解读：两条路线为什么各有不一致

**路线 A 的 4 个不一致（表 4 三列、表 5 列 2）全部来自 N = 922,477 的结果变量**，即有缺失值、面板不平衡的回归。
<span style="color:#c0392b">**⚠ StatsPAI 不足 #8**</span>：`sp.etwfe(family="poisson")` 用**队列虚拟变量**（Mundlak/R `etwfe` 写法）代替**城市对固定效应**。
泊松模型下两者的点估计仅在**平衡面板**上等价；面板不平衡时就不再相同（如表 4 列 (1)：0.1829 vs 论文 0.1785），而 Stata `jwdid ..., method(ppmlhdfe)` 吸收的是 `absorb(id_ci_cj year)`。
函数没有任何警告。路线 B（真正吸收城市对 FE）在这些列上系数完全一致，可以佐证。

**路线 B 的系数全部一致，但 N 不同，个别 SE 在第 4 位小数上差 1—2**：`sp.fepois`（pyfixest）会剔除结果全为零的城市对（分离观测），
Stata 在本数据全样本上却报告 `num_separated = 0`，把 71 万个全零观测计入 N（923,076）和聚类数（43,956）。
全零组不影响 $\hateta$，但聚类数 $G$ 变小，改变了 $G/(G-1)$ 小样本因子，SE 在第 4 位小数上略有差异（如表 1 列 (2)：0.2020 vs 0.2019）。
这是 **N/聚类口径差异**，不是估计错误；但 StatsPAI 没有“保留分离组以匹配 Stata 报告口径”的选项，也不在结果里报告剔除了多少观测（<span style="color:#c0392b">**⚠ 不足 #9**</span>）。

> 附注：Stata 自身在这一点上也不一致——同样的命令，在 5% 子样本上就剔除了 35,217 个分离观测（见第 9 节）。

## 9. StatsPAI 不足的现场演示

以下调用对应论文**图与附录**中用到的 `jwdid` 选项（本 notebook 不复现这些图表，但记录 StatsPAI 是否支持，便于升级）。
每个调用都用 `try/except` 包住，直接打印 StatsPAI 的报错。

In [11]:
d = df0.copy(); d["_y"] = d["合作发明专利"].astype(float)
d["_ns"] = d["citypair_ns"]
demos = {
    "#3 jwdid ..., never（图 3 事件研究：never-treated 对照 + 前期系数）":
        dict(cgroup="nevertreated"),
    "#4 jwdid y i.over, ... + estat simple, over()（图 4/6/7 分组异质性）":
        dict(xvar="_ns"),
}
for label, kw in demos.items():
    try:
        sp.etwfe(d, y="_y", group="id_ci_cj", time="year", first_treat="first_treat_lag", family="poisson", **kw)
        print("✔ 支持：", label)
    except Exception as e:
        print("✘ 不支持：", label, "\n   →", type(e).__name__, ":", str(e).splitlines()[0])

import inspect
params = inspect.signature(sp.etwfe).parameters
for opt, use in [("hettype", "#5 jwdid hettype(event/cohort/twfe)（附表 3 异质性设定）"),
                 ("scale", "#1 link/response 尺度选择"),
                 ("exovar", "#6 jwdid exovar()（附图 5 竞争性政策控制，外生控制不与处理交互）")]:
    print(("✔ 有参数 " if opt in params else "✘ 无参数 ") + f"`{opt}`：{use}")
print("\nsp.etwfe 现有参数：", list(params))

✘ 不支持： #3 jwdid ..., never（图 3 事件研究：never-treated 对照 + 前期系数） 
   → MethodIncompatibility : etwfe(family='poisson') does not support cgroup='nevertreated' yet.
✘ 不支持： #4 jwdid y i.over, ... + estat simple, over()（图 4/6/7 分组异质性） 
   → MethodIncompatibility : etwfe(family='poisson') does not support xvar='_ns' yet.
✘ 无参数 `hettype`：#5 jwdid hettype(event/cohort/twfe)（附表 3 异质性设定）
✘ 无参数 `scale`：#1 link/response 尺度选择
✘ 无参数 `exovar`：#6 jwdid exovar()（附图 5 竞争性政策控制，外生控制不与处理交互）

sp.etwfe 现有参数： ['data', 'y', 'group', 'time', 'first_treat', 'controls', 'cluster', 'alpha', 'xvar', 'panel', 'cgroup', 'family', 'weights', 'agg_weights']


<span style="color:#c0392b">**⚠ StatsPAI 不足 #7（静默错误 + 性能）**</span>：`sp.ppmlhdfe` **不剔除分离观测**（`separation=True` 只警告不删除）。
后果：(a) 全零城市对的 FE 趋向 −∞，IRLS 极慢：全样本表 1 列 (1) 单个回归 **86 分钟 CPU 仍未结束**，被手动终止（同一回归 Stata 约 40 秒、`sp.fepois` 约 2—17 秒、`sp.etwfe` 约 6—9 秒）；
(b) 下面的子样本中，点估计与 Stata/`sp.fepois` 在第 4 位小数上不同（相对差约 1e-3），且**不报错**，疑为未完全收敛。

下面在 5% 城市对子样本（2,198 个城市对）上对照 Stata。Stata 参考值由同一子样本 `ppmlhdfe 合作专利 tr_*, absorb(id_ci_cj year) cluster(id_ci_cj) keepsingletons` 得到：
N = 10,941（剔除分离 35,217 个观测），聚类 521。

In [12]:
sub_ids = df0["id_ci_cj"].drop_duplicates().sample(frac=0.05, random_state=1)
s = df0[df0["id_ci_cj"].isin(sub_ids)].copy(); s["_y"] = s["合作专利"].astype(float)
cs = tr_cells(s)
t0 = time.time(); r_fe = sp.fepois("_y ~ " + " + ".join(cs) + " | id_ci_cj + year", data=s, vcov={"CRV1": "id_ci_cj"}); t_fe = time.time() - t0
t0 = time.time(); r_pp = sp.ppmlhdfe(data=s, y="_y", x=cs, absorb="id_ci_cj + year", cluster="id_ci_cj"); t_pp = time.time() - t0
stata_sub = pd.Series([.02278347, .28755924, .18115765, .32113014, .44167148, .53361505, .40020514, .49944101, .74132133], index=cs)
print(f"5% 子样本 {len(s):,} 行：sp.fepois {t_fe:.1f}s | sp.ppmlhdfe {t_pp:.1f}s（{t_pp/t_fe:.0f}×）；Stata ppmlhdfe < 1s")
print(f"N：Stata 10,941 | sp.fepois {r_fe.data_info['nobs']:,} | sp.ppmlhdfe {r_pp.data_info['nobs']:,}")
print(f"聚类数：Stata 521 | sp.fepois {r_fe.model_info.get('n_clusters')} | sp.ppmlhdfe {r_pp.data_info.get('n_clusters')}")
pd.DataFrame({"Stata b": stata_sub, "sp.fepois b": r_fe.params[cs], "sp.ppmlhdfe b": r_pp.params[cs],
              "fepois−Stata": r_fe.params[cs] - stata_sub, "ppmlhdfe−Stata": r_pp.params[cs] - stata_sub}).style.format("{:.6f}").format("{:.1e}", subset=["fepois−Stata", "ppmlhdfe−Stata"])

5% 子样本 46,158 行：sp.fepois 0.1s | sp.ppmlhdfe 299.4s（3100×）；Stata ppmlhdfe < 1s
N：Stata 10,941 | sp.fepois 10,941 | sp.ppmlhdfe 46,158
聚类数：Stata 521 | sp.fepois 521 | sp.ppmlhdfe 2198


,Stata b,sp.fepois b,sp.ppmlhdfe b,fepois−Stata,ppmlhdfe−Stata
tr_2015_2015,0.022783,0.022783,0.022766,-4.1e-10,-1.7e-05
tr_2015_2016,0.287559,0.287559,0.287325,-2.7e-09,-2.3e-04
tr_2015_2017,0.181158,0.181158,0.181013,-3.8e-09,-1.4e-04
tr_2015_2018,0.321130,0.321130,0.320857,-1.3e-09,-2.7e-04
tr_2015_2019,0.441671,0.441671,0.441288,-4.3e-09,-3.8e-04
tr_2015_2020,0.533615,0.533615,0.533142,9.9e-10,-4.7e-04
tr_2018_2018,0.400205,0.400205,0.400100,-3.8e-10,-1.0e-04
tr_2018_2019,0.499441,0.499441,0.499281,4.2e-09,-1.6e-04
tr_2018_2020,0.741321,0.741321,0.741116,4.0e-09,-2.1e-04


## 10. 结论与 StatsPAI 升级清单

**复现程度**（21 个回归，4 位小数）：
- 路线 A `sp.etwfe(family="poisson")` + 手工 xb 聚合：**17/21** 系数、SE、N 全部一致；其余 4 个是不平衡面板（不足 #8）。
- 路线 B `sp.fepois` + 手工交互项 + `.lincom()`：**21/21 系数一致**，SE 在个别列的第 4 位小数上差 1—2；N 口径与 Stata 不同（不足 #9）。
- 直接用 `sp.etwfe` 的 headline：**0/21**（口径不同，不足 #1）。

**没有一条路线能“一行调用就得到论文数字”**，都需要手工补一步。目前最稳妥的 StatsPAI 写法是路线 B。

| # | 严重度 | 问题 | 论文中涉及 | 建议升级 |
|---|---|---|---|---|
| 1 | **高（口径）** | `sp.etwfe(family="poisson")` headline 只给响应尺度 AME（件数），无 xb/link 尺度 simple ATT；直接用 `r.estimate` 得 0.1419 而非论文 0.1691 | 表 1—5 全部 | 增加 `scale="link"`；在结果中同时报告两种尺度，并在 `summary()` 标注单位 |
| 2 | 中 | 非线性 ETWFE 无公开聚合 API（simple / group / calendar / event），需从 `model_info` 手工 delta 法；`model_info` 不含 $N_{gt}$ | 表 1—5；图 3 | `r.aggregate(type=..., scale=...)`，对标 `jwdid estat` / `etwfe::emfx` |
| 3 | 高（功能） | 泊松分支不支持 `cgroup="nevertreated"`（jwdid `never`：never-treated 对照、含前期系数的事件研究） | 图 3 动态诊断、附图 7 | 泊松分支实现 never 设计及 event 聚合 |
| 4 | 高（功能） | 泊松分支不支持 `xvar`（jwdid `y i.over` + `estat simple, over()` 分组 ATT） | 图 4、6、7；附图 9、10 | 实现 `xvar`/`over` 分组聚合 |
| 5 | 中 | 无 `hettype(event/cohort/twfe)` 受限异质性设定 | 附表 3 | 增加 `hettype` 参数 |
| 6 | 中 | 无 jwdid `exovar()`（不与处理交互的外生控制） | 附图 5 | 增加 `exovar` 或区分 `controls` 是否交互 |
| 7 | **最高（静默错误 + 性能）** | `sp.ppmlhdfe` 不剔除分离观测：子样本点估计与 Stata 差 1e-4 量级、N/聚类数偏大，且不报错；全样本 >86 分钟未完成（Stata 40 秒、`sp.fepois` 17 秒） | 任何含全零组的 PPML | 默认按 Correia–Guimarães–Zylkin 规则剔除分离观测（FE + simplex/ReLU），报告 `num_separated`；高维 FE 默认走快速后端 |
| 8 | **高（静默错误）** | `sp.etwfe(family="poisson")` 用队列虚拟变量代替个体 FE；平衡面板时点估计与 `jwdid`/`ppmlhdfe` 一致，**面板不平衡时（结果变量有缺失）静默不同**，差异达 0.005（表 4、表 5 列 2），无警告 | 表 4 全部、表 5 列 2 | 泊松分支改为吸收个体 FE（`absorb(unit time)`，可调用 fepois 后端），或检测到不平衡面板时报警 |
| 9 | 低 | `sp.fepois` 剔除全零 FE 组后 N、聚类数与 Stata 全样本报告口径不同（β 不变，SE 第 4 位小数可能不同），结果中无“剔除了多少观测”字段 | 表 1—5 的 N 行 | 报告 `num_separated`、`N_full`；可选保留分离组以对齐 Stata |

各项证据：#1/#2 见第 2 节；#3—#6 见第 9 节；#7 见第 9 节子样本对照；#8、#9 见第 8.1 节。